# meinRad Mainz: live bike-sharing snapshots

This notebook visualizes station-level `meinRad` availability snapshots for Mainz. It can read existing CSV snapshots, collect new snapshots, and create interactive maps and time animations when several snapshots are available.

**Learning goals**

- Load repeated live API snapshots as a time-dependent geospatial dataset.
- Map current station availability and floating bikes.
- Animate changes over time with a time slider.
- Identify empty, full, and highly variable stations.

In [ ]:
# Colab/Jupyter setup
!pip -q install pandas folium branca plotly

In [ ]:
import csv
import json
from pathlib import Path
import time
from datetime import datetime, timezone
from urllib.request import Request, urlopen

import branca.colormap as cm
import folium
import pandas as pd
from folium.plugins import MarkerCluster, TimestampedGeoJson
import plotly.express as px
pd.set_option("display.max_columns", 80)

## 1. Paths and optional live collection

The notebook writes snapshots to a local `data_loaders/meinrad/data/` folder. In Colab this folder is temporary, so rerun the collection cells when the runtime restarts.

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data_loaders" / "meinrad").exists():
    # Common Colab location after: !git clone https://github.com/yfeng-hsm/KI_Geodatenanalyse_SS26.git
    candidate = Path("/content/KI_Geodatenanalyse_SS26")
    if (candidate / "data_loaders" / "meinrad").exists():
        ROOT = candidate

DATA_DIR = ROOT / "data_loaders" / "meinrad" / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Snapshot folder:", DATA_DIR)

In [ ]:
# Set this to True when you want the notebook to collect new live snapshots.
# For a real temporal animation, collect every 5-15 minutes over several hours or days.
COLLECT_NEW_SNAPSHOTS = False
N_SNAPSHOTS = 3
WAIT_SECONDS = 30

NEXTBIKE_URL = "https://api.nextbike.net/maps/nextbike-live.json?domains=mz"


def fetch_meinrad_live_data():
    request = Request(
        NEXTBIKE_URL,
        headers={
            "Accept": "application/json",
            "User-Agent": "KI_Geodatenanalyse_SS26 meinRad teaching notebook",
        },
    )
    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def get_mainz_city(payload, city_uid=755):
    for country in payload.get("countries", []):
        for city in country.get("cities", []):
            if city.get("uid") == city_uid:
                return country, city
    raise ValueError("Mainz city uid 755 not found. The notebook expects the domains=mz feed.")


def flatten_places(country, city, collected_at_utc):
    rows = []
    for place in city.get("places", []):
        rows.append(
            {
                "collected_at_utc": collected_at_utc,
                "system_name": country.get("name"),
                "domain": country.get("domain"),
                "city_uid": city.get("uid"),
                "city_name": city.get("name"),
                "place_uid": place.get("uid"),
                "place_number": place.get("number"),
                "name": place.get("name"),
                "address": place.get("address"),
                "lat": place.get("lat"),
                "lng": place.get("lng"),
                "is_station": bool(place.get("spot")),
                "is_floating_bike": bool(place.get("bike")),
                "active_place": place.get("active_place"),
                "maintenance": bool(place.get("maintenance")),
                "terminal_type": place.get("terminal_type"),
                "place_type": place.get("place_type"),
                "booked_bikes": place.get("booked_bikes"),
                "bikes": place.get("bikes"),
                "bikes_available_to_rent": place.get("bikes_available_to_rent"),
                "bike_racks": place.get("bike_racks"),
                "free_racks": place.get("free_racks"),
                "special_racks": place.get("special_racks"),
                "free_special_racks": place.get("free_special_racks"),
                "bike_types_json": json.dumps(place.get("bike_types", {}), ensure_ascii=False, sort_keys=True),
                "bike_count_from_list": len(place.get("bike_list", [])),
            }
        )
    return rows


def collect_meinrad_snapshot(output_dir=DATA_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)
    collected_at = datetime.now(timezone.utc)
    collected_at_utc = collected_at.isoformat(timespec="seconds")
    stamp = collected_at.strftime("%Y%m%dT%H%M%SZ")
    payload = fetch_meinrad_live_data()
    country, city = get_mainz_city(payload)
    rows = flatten_places(country, city, collected_at_utc)
    if not rows:
        raise ValueError("No places returned by the meinRad feed.")
    csv_path = output_dir / f"meinrad_mainz_places_{stamp}.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Collected {len(rows)} places -> {csv_path}")
    return csv_path


if COLLECT_NEW_SNAPSHOTS:
    for i in range(N_SNAPSHOTS):
        print(f"Collecting snapshot {i + 1}/{N_SNAPSHOTS} ...")
        collect_meinrad_snapshot(DATA_DIR)
        if i < N_SNAPSHOTS - 1:
            time.sleep(WAIT_SECONDS)

## 2. Load all station snapshots

In [ ]:
snapshot_files = sorted(DATA_DIR.glob("meinrad_mainz_places_*.csv"))
if not snapshot_files:
    print(f"No snapshot CSV files found in {DATA_DIR}. Collecting one live snapshot now ...")
    collect_meinrad_snapshot(DATA_DIR)
    snapshot_files = sorted(DATA_DIR.glob("meinrad_mainz_places_*.csv"))

df = pd.concat((pd.read_csv(path) for path in snapshot_files), ignore_index=True)
df["collected_at_utc"] = pd.to_datetime(df["collected_at_utc"], utc=True)
df["snapshot_label"] = df["collected_at_utc"].dt.strftime("%Y-%m-%d %H:%M UTC")

numeric_cols = [
    "lat",
    "lng",
    "bikes_available_to_rent",
    "bikes",
    "bike_racks",
    "free_racks",
    "bike_count_from_list",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

stations = df[df["is_station"] == True].copy()
floating = df[df["is_floating_bike"] == True].copy()

print(f"Loaded {len(snapshot_files)} snapshot file(s)")
print(f"Rows: {len(df):,}; stations: {len(stations):,}; floating-bike rows: {len(floating):,}")
print("Snapshot times:")
display(df[["collected_at_utc", "snapshot_label"]].drop_duplicates().sort_values("collected_at_utc"))

## 3. Current station availability map

The latest snapshot is shown first. Circle size represents available bikes; color represents station fill ratio. Click any station for details.

In [ ]:
latest_time = df["collected_at_utc"].max()
latest_stations = stations[stations["collected_at_utc"] == latest_time].copy()
latest_stations["fill_ratio"] = (
    latest_stations["bikes_available_to_rent"]
    / latest_stations["bike_racks"].where(latest_stations["bike_racks"] > 0)
)
latest_stations["marker_size"] = latest_stations["bikes_available_to_rent"].clip(lower=1)

center = [latest_stations["lat"].mean(), latest_stations["lng"].mean()]
fill_colormap = cm.LinearColormap(
    colors=["#d73027", "#fee08b", "#1a9850"],
    vmin=0,
    vmax=1,
    caption="Station fill ratio: available bikes / racks",
)

latest_map = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
cluster = MarkerCluster(name="meinRad stations").add_to(latest_map)

for _, row in latest_stations.iterrows():
    fill_ratio = row["fill_ratio"]
    if pd.isna(fill_ratio):
        fill_ratio = 0
    popup_html = f"""
    <b>{row['name']}</b><br>
    Available bikes: {int(row['bikes_available_to_rent'])}<br>
    Bike racks: {int(row['bike_racks']) if pd.notna(row['bike_racks']) else 'n/a'}<br>
    Free racks: {int(row['free_racks']) if pd.notna(row['free_racks']) else 'n/a'}<br>
    Maintenance: {row['maintenance']}<br>
    Time: {latest_time:%Y-%m-%d %H:%M UTC}
    """
    folium.CircleMarker(
        location=[row["lat"], row["lng"]],
        radius=max(4, min(18, 4 + float(row["bikes_available_to_rent"]) ** 0.5 * 2.2)),
        color="#253494",
        weight=1,
        fill=True,
        fill_color=fill_colormap(fill_ratio),
        fill_opacity=0.78,
        tooltip=f"{row['name']}: {int(row['bikes_available_to_rent'])} bikes",
        popup=folium.Popup(popup_html, max_width=320),
    ).add_to(cluster)

fill_colormap.add_to(latest_map)
folium.LayerControl().add_to(latest_map)
latest_map

## 4. Dynamic map over time

This Folium/Leaflet map uses a time slider. It becomes useful after several snapshots have been collected.

In [ ]:
snapshot_count = stations["collected_at_utc"].nunique()

if snapshot_count < 2:
    print(
        "Only one snapshot is available. Set COLLECT_NEW_SNAPSHOTS=True above, "
        "or run collect_meinrad_snapshot(DATA_DIR) repeatedly, then rerun this cell for animation."
    )
else:
    features = []
    max_bikes = max(1, int(stations["bikes_available_to_rent"].max()))
    bike_colormap = cm.LinearColormap(
        colors=["#d73027", "#fee08b", "#1a9850"],
        vmin=0,
        vmax=max_bikes,
        caption="Available bikes",
    )

    for _, row in stations.dropna(subset=["lat", "lng"]).iterrows():
        bikes = int(row["bikes_available_to_rent"]) if pd.notna(row["bikes_available_to_rent"]) else 0
        radius = max(5, min(18, 5 + bikes ** 0.5 * 2.0))
        features.append(
            {
                "type": "Feature",
                "geometry": {"type": "Point", "coordinates": [row["lng"], row["lat"]]},
                "properties": {
                    "time": row["collected_at_utc"].isoformat(),
                    "popup": (
                        f"<b>{row['name']}</b><br>"
                        f"Available bikes: {bikes}<br>"
                        f"Bike racks: {row['bike_racks']}<br>"
                        f"Free racks: {row['free_racks']}<br>"
                        f"Time: {row['snapshot_label']}"
                    ),
                    "tooltip": f"{row['name']}: {bikes} bikes",
                    "icon": "circle",
                    "iconstyle": {
                        "fillColor": bike_colormap(bikes),
                        "fillOpacity": 0.75,
                        "stroke": "true",
                        "color": "#253494",
                        "weight": 1,
                        "radius": radius,
                    },
                    "style": {"color": "#253494"},
                },
            }
        )

    time_map = folium.Map(location=center, zoom_start=12, tiles="CartoDB positron")
    TimestampedGeoJson(
        {"type": "FeatureCollection", "features": features},
        period="PT1M",
        duration="PT20M",
        add_last_point=False,
        auto_play=False,
        loop=False,
        max_speed=2,
        loop_button=True,
        date_options="YYYY-MM-DD HH:mm:ss",
        time_slider_drag_update=True,
    ).add_to(time_map)
    bike_colormap.add_to(time_map)
    time_map

## 5. Station imbalance and change

In [ ]:
latest_view = latest_stations[
    ["name", "bikes_available_to_rent", "bike_racks", "free_racks", "maintenance"]
].sort_values("bikes_available_to_rent")

display(latest_view.head(15).rename(columns={"bikes_available_to_rent": "available_bikes"}))

top_available = latest_view.sort_values("bikes_available_to_rent", ascending=False).head(15)
fig_bar = px.bar(
    top_available.sort_values("bikes_available_to_rent"),
    x="bikes_available_to_rent",
    y="name",
    orientation="h",
    title="Stations with the most available bikes in the latest snapshot",
    labels={"bikes_available_to_rent": "Available bikes", "name": "Station"},
    height=500,
)
fig_bar.show()

In [ ]:
if snapshot_count < 2:
    print("Change metrics need at least two snapshots.")
else:
    station_variation = (
        stations.groupby(["place_uid", "name"], as_index=False)
        .agg(
            min_available=("bikes_available_to_rent", "min"),
            max_available=("bikes_available_to_rent", "max"),
            mean_available=("bikes_available_to_rent", "mean"),
            snapshots=("collected_at_utc", "nunique"),
        )
    )
    station_variation["range_available"] = (
        station_variation["max_available"] - station_variation["min_available"]
    )
    display(station_variation.sort_values("range_available", ascending=False).head(20))

    top_dynamic_ids = station_variation.sort_values("range_available", ascending=False).head(12)["place_uid"]
    top_dynamic = stations[stations["place_uid"].isin(top_dynamic_ids)]
    fig_lines = px.line(
        top_dynamic,
        x="collected_at_utc",
        y="bikes_available_to_rent",
        color="name",
        markers=True,
        title="Most variable stations across collected snapshots",
        labels={"collected_at_utc": "Time", "bikes_available_to_rent": "Available bikes"},
        height=550,
    )
    fig_lines.show()

## 6. Export an interactive HTML map

In [ ]:
EXPORT_HTML = False

if EXPORT_HTML:
    output_path = DATA_DIR / "meinrad_mainz_latest_map.html"
    latest_map.save(output_path)
    print("Wrote", output_path)